<div style="background: linear-gradient(135deg, #0B1F3F 0%, #008C8C 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
<h1 style="color: white; font-family: Georgia, serif; margin: 0; font-size: 2.4em;">
🔬 Session 14: Signal Extraction
</h1>
<p style="color: #B8953E; font-size: 1.3em; margin-top: 10px; font-family: Calibri, sans-serif;">
Correlation Screening • Variance Filters • PCA vs Factor Analysis
</p>
<p style="color: #B0D0D0; font-size: 0.95em; margin-top: 8px;">
Credit Risk Modelling Programme &nbsp;|&nbsp; MBA Advanced Analytics &nbsp;|&nbsp; 2025–26
</p>
</div>

<div style="background: #FFFFFF; border: 2px solid #B8953E; padding: 20px 28px; border-radius: 10px; margin: 10px 0 20px 0;">
<h3 style="color: #0B1F3F; margin-top: 0;">📖 Where We Are in the Journey</h3>
<p style="color: #333; font-size: 1.05em; line-height: 1.7;">
In <strong>Sessions 13 & 13B</strong>, we built a deep, intuitive understanding of the data: its structure,
its quality issues, and which features appear to predict default. We have a Feature Scorecard ranking
features by apparent signal strength.
</p>
<p style="color: #333; font-size: 1.05em; line-height: 1.7;">
Now we <strong>formalise</strong> that intuition. This session answers three questions:<br>
<strong>1.</strong> Which features are <em>redundant</em> (telling us the same thing)?<br>
<strong>2.</strong> Which features are <em>uninformative</em> (telling us nothing at all)?<br>
<strong>3.</strong> Can we <em>compress</em> 120+ features into a smaller set without losing important information?
</p>
<p style="color: #555; font-size: 0.95em; margin-top: 12px; font-style: italic;">
⬆️ Increasing self-sufficiency: The code hints remain complete, but you’ll now encounter
<strong>“Think About It”</strong> and <strong>“What Would You Do Next?”</strong> prompts that ask you
to decide the analytical direction before we reveal it. This mirrors real-world analytics where
nobody hands you a roadmap.
</p>
</div>

<div style="background: #FFFFFF; border: 2px solid #008C8C; padding: 20px 28px; border-radius: 10px; margin: 10px 0 20px 0;">
<h3 style="color: #0B1F3F; margin-top: 0;">🗺️ Notebook Roadmap</h3>
<ol style="color: #333; font-size: 1.05em; line-height: 1.8;">
<li><strong>Setup & Data Preparation</strong> — Reload, clean, and prepare a numeric-only matrix</li>
<li><strong>Near-Zero Variance Filters</strong> — Remove features that carry almost no information</li>
<li><strong>Correlation Screening</strong> — Detect and handle multicollinearity</li>
<li><strong>PCA: Compressing the Feature Space</strong> — Reduce 100+ features to a handful of components</li>
<li><strong>Factor Analysis: Discovering Latent Constructs</strong> — Find the hidden themes in the data</li>
<li><strong>PCA vs FA: Head-to-Head Comparison</strong> — When to use which, and what each tells a manager</li>
<li><strong>Building the Reduced Feature Set</strong> — Make final decisions and prepare for modelling</li>
</ol>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 1: Setup & Data Preparation</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Preparing a clean numeric matrix for dimensionality reduction</p>
</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')

NAVY  = '#0B1F3F'
TEAL  = '#008C8C'
GOLD  = '#B8953E'
CORAL = '#E8634A'
LGOLD = '#FDF6E8'
LTEAL = '#E0F2F2'
PURPLE = '#6C5B7B'
PALETTE = [NAVY, TEAL, GOLD, CORAL, PURPLE, '#C06C84']
sns.set_palette(PALETTE)

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.titleweight': 'bold',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.4f}'.format)

print("\u2705 Libraries loaded (including sklearn & scipy).")

In [ ]:
# ── Update path to your actual data folder ──
DATA_DIR = r'C:\Users\DevA\OneDrive\Desktop\All in one - Desktop Files\MBA\Q8\AI ML\April 1 and 8'

app = pd.read_csv(os.path.join(DATA_DIR, 'application_train.csv'))

# DAYS_EMPLOYED anomaly (Session 13)
app['DAYS_EMPLOYED_ANOMALY'] = (app['DAYS_EMPLOYED'] == 365243).astype(int)
app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].replace(365243, np.nan)

# Derived KPIs (Session 13B)
app['AGE_YEARS']         = (-app['DAYS_BIRTH'] / 365).round(1)
app['EMPLOYMENT_YEARS']  = (-app['DAYS_EMPLOYED'] / 365).round(1)
app['DEBT_TO_INCOME']    = (app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']).round(2)
app['ANNUITY_BURDEN']    = (app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']).round(4)
app['CREDIT_GOODS_RATIO']= (app['AMT_CREDIT'] / app['AMT_GOODS_PRICE']).round(3)
app['EXT_SCORE_BLEND']   = app[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].mean(axis=1).round(4)

print(f"\u2705 Data loaded: {app.shape[0]:,} rows \u00d7 {app.shape[1]} columns")
print(f"   Default rate: {app['TARGET'].mean():.2%}")


<div style="background: linear-gradient(135deg, #FFF8E1, #FFF3E0); border: 2px solid #B8953E; padding: 16px 20px; border-radius: 8px; margin: 16px 0;">
<strong style="color: #B8953E;">🤔 Think About It</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">Before we can run PCA or compute correlations, we need a <em>clean numeric matrix</em>: no categorical columns, no excessive missingness, and standardised scales. What steps would you take to prepare this? Write them down before scrolling further.</p>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 1: Prepare the Numeric Feature Matrix</h3>
</div>

In [ ]:
from sklearn.impute import SimpleImputer

# Step 1: Select numeric columns, exclude TARGET
target = app['TARGET'].copy()
num_df = app.select_dtypes(include=[np.number]).drop(columns=['TARGET'], errors='ignore')

print(f"Step 1 — Numeric columns selected : {num_df.shape[1]}")

# Step 2: Drop columns with >40% missing
miss_pct = num_df.isnull().mean()
high_miss_cols = miss_pct[miss_pct > 0.40].index.tolist()
num_df = num_df.drop(columns=high_miss_cols)
print(f"Step 2 — After dropping >40% missing: {num_df.shape[1]} columns ({len(high_miss_cols)} dropped)")
print(f"         Dropped: {high_miss_cols[:8]}{'...' if len(high_miss_cols) > 8 else ''}")

# Step 3: Impute remaining NaN with median
imputer = SimpleImputer(strategy='median')
numeric_imputed_arr = imputer.fit_transform(num_df)
numeric_imputed = pd.DataFrame(numeric_imputed_arr, columns=num_df.columns)
print(f"Step 3 — Median imputation applied. NaN remaining: {numeric_imputed.isnull().sum().sum()}")

# Step 4: Standardise
scaler = StandardScaler()
numeric_scaled_arr = scaler.fit_transform(numeric_imputed)
numeric_scaled = pd.DataFrame(numeric_scaled_arr, columns=num_df.columns)
print(f"Step 4 — Standardised. Mean≈0, Std≈1 verified.")
print(f"\n✅ Clean numeric matrix: {numeric_scaled.shape[0]:,} rows × {numeric_scaled.shape[1]} features")


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Select numeric: <code>app.select_dtypes(include=[np.number])</code>. Find high-missingness: <code>miss_pct = df.isnull().mean(); drop = miss_pct[miss_pct > 0.4].index</code>. Impute: <code>SimpleImputer(strategy='median').fit_transform(df)</code>. Scale: <code>StandardScaler().fit_transform(df)</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;">We started with ~108 numeric columns. After dropping high-missingness columns (mainly housing features like APARTMENTS_AVG, BASEMENTAREA_MEDI, etc.), we’re left with roughly 80–90 features. These are now imputed and standardised — ready for variance filters, correlation screening, and PCA.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 2: Near-Zero Variance Filters</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Removing features that carry almost no information</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Identify and remove features with near-zero variance. If a column has the same value for 99% of applicants, it cannot help distinguish defaulters from non-defaulters.</span>
</div>

<div style="background: #E0F2F2; border: 2px solid #008C8C; padding: 16px 22px; border-radius: 8px; margin: 12px 0;">
<strong style="color: #008C8C;">📐 What Is Near-Zero Variance?</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">
Imagine a column where 99.5% of values are 0 and 0.5% are 1. Technically it has two unique values,
but it’s <em>almost</em> constant. For most models, this feature adds noise rather than signal.
<br><br>
We measure this with two metrics:<br>
• <strong>Variance</strong> — after standardisation, truly constant columns have variance = 0.<br>
• <strong>Frequency ratio</strong> — (most common value count) / (second most common). If this ratio > 20,
the feature is dominated by a single value.
</p>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 2: Identify Near-Zero Variance Features</h3>
</div>

In [ ]:
# Near-Zero Variance Detection
nzv_report = []
for col in numeric_imputed.columns:
    vc = numeric_imputed[col].value_counts()
    variance = numeric_imputed[col].var()
    n_unique = numeric_imputed[col].nunique()
    pct_most_common = vc.iloc[0] / len(numeric_imputed) * 100
    freq_ratio = vc.iloc[0] / vc.iloc[1] if len(vc) > 1 else float('inf')
    pct_unique = n_unique / len(numeric_imputed) * 100
    is_nzv = (freq_ratio > 20) and (pct_unique < 1.0)
    nzv_report.append({
        'Feature': col,
        'Variance': round(variance, 4),
        'N Unique': n_unique,
        'Freq Ratio': round(freq_ratio, 1),
        '% Most Common': round(pct_most_common, 2),
        '% Unique': round(pct_unique, 4),
        'NZV Flag': is_nzv
    })

nzv_df = pd.DataFrame(nzv_report).sort_values('Freq Ratio', ascending=False)
nzv_cols = nzv_df[nzv_df['NZV Flag']]['Feature'].tolist()

print(f"Near-Zero Variance features found: {len(nzv_cols)}")
print()
print(nzv_df[nzv_df['NZV Flag']][['Feature','Freq Ratio','% Most Common','% Unique']].to_string(index=False))


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Frequency ratio: <code>vc = data.value_counts(); fr = vc.iloc[0] / vc.iloc[1]</code>. Most common %: <code>vc.iloc[0] / len(data) * 100</code>. Flag: <code>(freq_ratio > 20) & (pct_unique < 1)</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;">Most NZV features are the <strong>FLAG_DOCUMENT_*</strong> columns — binary flags where 99%+ of values are 0. These record whether the applicant provided specific documents. Since almost nobody provides these documents, the column can’t discriminate between good and bad borrowers. We’ll remove them to reduce noise.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 3: Remove NZV Features and Assess Impact</h3>
</div>

In [ ]:
# Remove NZV Features
print(f"Dropping {len(nzv_cols)} NZV columns:")
for c in nzv_cols:
    print(f"  - {c}")

numeric_imputed_filtered = numeric_imputed.drop(columns=nzv_cols)
numeric_scaled_filtered  = numeric_scaled.drop(columns=nzv_cols)

print(f"\nBefore: {numeric_imputed.shape[1]} features")
print(f"After : {numeric_imputed_filtered.shape[1]} features")
print(f"Removed: {len(nzv_cols)} NZV features (mostly FLAG_DOCUMENT_* columns)")


<div style="background: linear-gradient(135deg, #FFF8E1, #FFF3E0); border: 2px solid #B8953E; padding: 16px 20px; border-radius: 8px; margin: 16px 0;">
<strong style="color: #B8953E;">🤔 Think About It</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">We’ve removed uninformative features. But we haven’t yet addressed <em>redundant</em> features — pairs of columns that tell us essentially the same thing. What analysis technique from Session 13B already gave us a clue about redundancy? What threshold would you use to call two features “redundant”?</p>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 3: Correlation Screening</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Detecting and handling multicollinearity</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Identify pairs of features that are highly correlated (|r| > 0.8) and decide which one to keep. Understand why multicollinearity is a problem for some models but not others.</span>
</div>

<div style="background: #E0F2F2; border: 2px solid #008C8C; padding: 16px 22px; border-radius: 8px; margin: 12px 0;">
<strong style="color: #008C8C;">📐 Why Does Multicollinearity Matter?</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">
If two features are highly correlated (e.g., AMT_CREDIT and AMT_GOODS_PRICE at r=0.97), they carry
nearly the same information. Including both causes two problems:<br><br>
• <strong>For logistic regression:</strong> Coefficient estimates become unstable. A small change in data
can flip the sign of a coefficient, making interpretation impossible.<br>
• <strong>For PCA:</strong> The correlated pair will dominate the first principal component, wasting variance
on redundancy instead of true signal.<br>
• <strong>For tree-based models (XGBoost):</strong> Multicollinearity is <em>not</em> a problem. Trees
can pick whichever feature happens to split best. But keeping redundant features still wastes memory
and slows training.
</p>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 4: Identify Highly Correlated Pairs</h3>
</div>

In [ ]:
# Correlation Screening — Find Highly Correlated Pairs
threshold = 0.80

corr_matrix = numeric_imputed_filtered.corr().abs()

# Upper triangle only
mask = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
upper = corr_matrix.where(mask)

# Stack and filter
high_corr = (upper.stack()
             .reset_index()
             .rename(columns={'level_0':'Feature_A', 'level_1':'Feature_B', 0:'Correlation'})
             .query(f'Correlation > {threshold}')
             .sort_values('Correlation', ascending=False)
             .reset_index(drop=True))

high_corr_df = high_corr.copy()
print(f"Highly correlated pairs (|r| > {threshold}): {len(high_corr_df)}")
print()
print(high_corr_df.head(20).to_string(index=False))


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Upper triangle mask: <code>np.triu(np.ones(corr.shape), k=1).astype(bool)</code>. Apply it: <code>upper = corr.where(mask)</code>. Then loop through or use <code>upper.stack()</code> to find pairs above threshold.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;">You should find pairs like AMT_CREDIT ↔ AMT_GOODS_PRICE (r≈0.97), several DAYS_* columns that are near-duplicates, and derived KPIs that are mathematically linked to their inputs. The question now is: which one in each pair do we keep?</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 5: Decide Which Features to Keep</h3>
</div>

In [ ]:
# Drop the weaker feature in each correlated pair
# Keep whichever is more correlated with TARGET

target_corrs = numeric_imputed_filtered.corrwith(target).abs()

to_drop = set()
for _, row in high_corr_df.iterrows():
    feat_a, feat_b = row['Feature_A'], row['Feature_B']
    if feat_a in to_drop or feat_b in to_drop:
        continue
    corr_a = target_corrs.get(feat_a, 0)
    corr_b = target_corrs.get(feat_b, 0)
    weaker = feat_b if corr_a >= corr_b else feat_a
    to_drop.add(weaker)

print(f"Features to drop (redundant, weaker TARGET correlation): {len(to_drop)}")
for f in sorted(to_drop):
    print(f"  - {f}")

# Apply drops
numeric_decorr      = numeric_imputed_filtered.drop(columns=list(to_drop))
numeric_decorr_scaled = numeric_scaled_filtered.drop(columns=list(to_drop))

print(f"\nFeatures after NZV removal   : {numeric_imputed_filtered.shape[1]}")
print(f"Features after corr screening: {numeric_decorr.shape[1]}")
print(f"Total reduction from original: {numeric_imputed.shape[1]} → {numeric_decorr.shape[1]}")


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Compute each feature’s TARGET correlation: <code>target_corrs = numeric_imputed_filtered.corrwith(target).abs()</code>. Use a set <code>to_drop</code> to track removals. Skip pairs where one is already dropped.</span>
</div>

<div style="background: #F0E6F6; border-left: 5px solid #6C5B7B; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #6C5B7B;">👔 Manager’s Take</strong><br>
<span style="color: #333;">When a regulator asks “why did you drop AMT_GOODS_PRICE but keep AMT_CREDIT?”, you need a defensible answer. “They carry 97% of the same information, and AMT_CREDIT is more directly correlated with default risk” is a strong response. Document your decisions.</span>
</div>

<div style="background: linear-gradient(90deg, #E8EDF4, #F0E6F6); border: 2px dashed #6C5B7B; padding: 16px 20px; border-radius: 8px; margin: 16px 0;">
<strong style="color: #6C5B7B;">🧭 What Would You Do Next?</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">We’ve removed uninformative features (NZV) and redundant features (correlation screening). But we still have perhaps 60–80 features. Some models struggle with this many dimensions. What technique could compress these into a smaller set while retaining the most information? And is “compression” the only way to think about dimensionality reduction?</p>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 4: PCA: Compressing the Feature Space</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Finding the directions of maximum variance</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Apply Principal Component Analysis (PCA) to the decorrelated feature set. Interpret the scree plot and the first few components. Understand what PCA does in plain language.</span>
</div>

<div style="background: linear-gradient(135deg, #E8EDF4, #E0F2F2); border: 2px solid #0B1F3F; padding: 20px 24px; border-radius: 10px; margin: 12px 0;">
<h3 style="color: #0B1F3F; margin-top: 0;">📐 PCA in Plain English</h3>
<p style="color: #333; line-height: 1.7;">
Imagine you have 80 columns of data. PCA asks: <em>“If I could only keep a few summary numbers
per applicant, which summaries would capture the most variation in the data?”</em>
</p>
<p style="color: #333; line-height: 1.7;">
Each “principal component” is a weighted combination of the original features. The first
component captures the single direction in which applicants differ the most. The second captures
the next-most-variable direction (perpendicular to the first), and so on.
</p>
<p style="color: #333; line-height: 1.7;">
<strong>Analogy:</strong> Think of photographing a building. You could take 80 photos from 80 angles.
But 3–4 well-chosen angles (front, side, top) capture almost everything. PCA finds those best angles
automatically.
</p>
<p style="color: #333; line-height: 1.7;">
<strong>Key limitation:</strong> PCA components are mathematical constructs. PC1 might be “0.3 × income
+ 0.25 × credit − 0.2 × age + ...” — a blend that’s hard to explain to a manager.
This is where Factor Analysis offers an alternative (Part 5).
</p>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 6: Run PCA and Interpret the Scree Plot</h3>
</div>

In [ ]:
# PCA — Scree Plot and Variance Analysis
pca_full = PCA(random_state=42)
pca_scores = pca_full.fit_transform(numeric_decorr_scaled)

evr = pca_full.explained_variance_ratio_
cumvar = np.cumsum(evr)

# Find thresholds
n_pca_80 = int(np.argmax(cumvar >= 0.80)) + 1
n_pca_90 = int(np.argmax(cumvar >= 0.90)) + 1
n_pca_95 = int(np.argmax(cumvar >= 0.95)) + 1

print(f"Components for 80% variance : {n_pca_80}")
print(f"Components for 90% variance : {n_pca_90}")
print(f"Components for 95% variance : {n_pca_95}")
print(f"Total features in matrix    : {numeric_decorr_scaled.shape[1]}")

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Individual variance (bar)
axes[0].bar(range(1, 31), evr[:30] * 100, color=TEAL, alpha=0.8, edgecolor='white')
axes[0].set_title('Individual Variance Explained — First 30 PCs',
                  color=NAVY, fontweight='bold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')

# Cumulative variance (line)
axes[1].plot(range(1, len(cumvar)+1), cumvar * 100,
             color=TEAL, linewidth=2.5, marker='o', markersize=3)
for pct, n, color in [(80, n_pca_80, GOLD), (90, n_pca_90, CORAL), (95, n_pca_95, PURPLE)]:
    axes[1].axhline(pct, color=color, linestyle='--', linewidth=1.3, label=f'{pct}% @ PC{n}')
    axes[1].axvline(n,   color=color, linestyle='--', linewidth=1.0)
axes[1].set_title('Cumulative Variance Explained', color=NAVY, fontweight='bold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_xlim(0, 60)
axes[1].legend(fontsize=10)

plt.suptitle('PCA Scree Plot — Home Credit Feature Matrix', fontsize=14, color=NAVY, fontweight='bold')
plt.tight_layout()
plt.show()


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Fit PCA: <code>pca = PCA(); pca.fit_transform(numeric_decorr)</code>. Variance per PC: <code>pca.explained_variance_ratio_</code>. Cumulative: <code>np.cumsum(pca.explained_variance_ratio_)</code>. Threshold index: <code>np.argmax(cumulative >= 0.90) + 1</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;">The scree plot typically shows that 20–30 components capture 90% of the variance in 80+ features. This is dramatic compression: we’ve gone from “need 80 numbers to describe each applicant” to “25 numbers capture 90% of the variation.” The first component alone might capture 8–10%. The “elbow” in the scree plot (where bars get tiny) shows where additional components add little.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 7: Interpret the First 3 Components</h3>
</div>

In [ ]:
# Interpret the First 3 PCA Components
n_show = 5   # number of PCs to extract
loadings = pd.DataFrame(
    pca_full.components_[:n_show].T,
    columns=[f'PC{i+1}' for i in range(n_show)],
    index=numeric_decorr_scaled.columns
)

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
for i, pc in enumerate(['PC1', 'PC2', 'PC3']):
    top = loadings[pc].abs().nlargest(12)
    vals = loadings.loc[top.index, pc]
    colors = [TEAL if v > 0 else CORAL for v in vals]
    axes[i].barh(top.index, vals, color=colors, edgecolor='white')
    axes[i].axvline(0, color=NAVY, linewidth=0.8)
    axes[i].set_title(f'{pc} — Top 12 Features\n(var explained: {pca_full.explained_variance_ratio_[i]:.2%})',
                      color=NAVY, fontweight='bold', fontsize=11)
    axes[i].set_xlabel('Loading')
    axes[i].tick_params(axis='y', labelsize=8)

plt.suptitle('PCA Component Loadings — What Does Each PC Represent?',
             fontsize=13, color=NAVY, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 5 features per component:")
for pc in ['PC1', 'PC2', 'PC3']:
    top5 = loadings[pc].abs().nlargest(5).index.tolist()
    print(f"  {pc}: {top5}")

# Extract PCA scores for 90% variance
pca_90 = PCA(n_components=n_pca_90, random_state=42)
pca_components_90 = pca_90.fit_transform(numeric_decorr_scaled)
pca_df = pd.DataFrame(pca_components_90,
                      columns=[f'PC{i+1}' for i in range(n_pca_90)])
pca_df['TARGET'] = target.values
print(f"\n✅ PCA matrix (90% variance): {pca_df.shape[0]:,} rows × {pca_df.shape[1]} columns")


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Loadings: <code>pd.DataFrame(pca.components_[:5].T, columns=[f'PC{i+1}' for i in range(5)], index=numeric_decorr.columns)</code>. Top features per PC: <code>loadings['PC1'].abs().nlargest(12)</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>Naming the components</strong> (your attempt — exact names depend on your data):<br>• <strong>PC1</strong> might be dominated by document flags and region ratings — call it “Application Completeness.”<br>• <strong>PC2</strong> might load heavily on financial amounts — “Loan Size.”<br>• <strong>PC3</strong> might blend EXT_SOURCE scores — “External Creditworthiness.”<br>But notice how each PC mixes many features. This is PCA’s weakness: components are hard to name.</span>
</div>

<div style="background: linear-gradient(135deg, #FFF8E1, #FFF3E0); border: 2px solid #B8953E; padding: 16px 20px; border-radius: 8px; margin: 16px 0;">
<strong style="color: #B8953E;">🤔 Think About It</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">PCA found efficient summaries, but they’re hard to explain. A manager asking “what drove this applicant’s score?” won’t accept “PC3 was high.” Is there a method that produces factors with cleaner, more interpretable loadings? What would “cleaner” mean?</p>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 5: Factor Analysis: Discovering Latent Constructs</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Finding the hidden themes in the data</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Apply Factor Analysis with varimax rotation to the same data. Compare the resulting factor loadings with PCA loadings. Give each factor a plain-English name.</span>
</div>

<div style="background: linear-gradient(135deg, #E8EDF4, #E0F2F2); border: 2px solid #0B1F3F; padding: 20px 24px; border-radius: 10px; margin: 12px 0;">
<h3 style="color: #0B1F3F; margin-top: 0;">📐 Factor Analysis in Plain English</h3>
<p style="color: #333; line-height: 1.7;">
Factor Analysis asks a different question from PCA. PCA asks “how do I compress?”
Factor Analysis asks: <em>“What hidden constructs explain the correlations among my features?”</em>
</p>
<p style="color: #333; line-height: 1.7;">
<strong>Analogy:</strong> Imagine you measure students’ grades in maths, physics, history, and literature.
PCA would give you “PC1 = 0.4×maths + 0.35×physics + 0.3×history + 0.25×literature.”
Factor Analysis would discover two latent factors: <em>“Scientific Aptitude”</em> (loads on maths
and physics) and <em>“Humanities Aptitude”</em> (loads on history and literature). Much cleaner.
</p>
<p style="color: #333; line-height: 1.7;">
The trick is <strong>rotation</strong>: after extracting factors, we rotate them so that each feature
loads heavily on <em>one</em> factor and weakly on others. This makes interpretation much easier.
The most common rotation is <strong>varimax</strong>.
</p>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 8: Run Factor Analysis</h3>
</div>

In [ ]:
from sklearn.decomposition import FactorAnalysis

# Use the same number of factors as PCA 80% threshold
n_factors = n_pca_80

fa = FactorAnalysis(n_components=n_factors, rotation='varimax', random_state=42)
fa_scores = fa.fit_transform(numeric_decorr_scaled)

# Build loadings DataFrame
fa_loadings = pd.DataFrame(
    fa.components_.T,
    index=numeric_decorr_scaled.columns,
    columns=[f'Factor_{i+1}' for i in range(n_factors)]
)

print(f"Factor Analysis complete: {n_factors} factors with varimax rotation")
print(f"Loadings matrix shape: {fa_loadings.shape}")
print()

# Top 5 features per factor
for col in fa_loadings.columns[:6]:
    top5 = fa_loadings[col].abs().nlargest(5)
    print(f"{col}: {top5.index.tolist()}")


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Fit: <code>fa = FactorAnalysis(n_components=n, rotation='varimax', random_state=42); fa.fit_transform(numeric_decorr)</code>. Loadings: <code>pd.DataFrame(fa.components_.T, columns=[...], index=features)</code>. Top per factor: <code>loadings['Factor_1'].abs().nlargest(5)</code>.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 9: Name Your Factors</h3>
</div>

In [ ]:
# Factor Loadings Heatmap + Factor Naming

# Select top features that load heavily on any factor (first 6 factors)
tops = set()
for col in fa_loadings.columns[:6]:
    tops.update(fa_loadings[col].abs().nlargest(8).index.tolist())
tops = list(tops)

# Sort features by their dominant factor
dominant = fa_loadings.loc[tops].abs().idxmax(axis=1).sort_values()
sorted_feats = dominant.index.tolist()

fig, ax = plt.subplots(figsize=(14, max(8, len(sorted_feats) * 0.35)))
sns.heatmap(fa_loadings.loc[sorted_feats, fa_loadings.columns[:6]],
            annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.4, ax=ax, cbar_kws={'label': 'Loading'},
            annot_kws={'size': 7})
ax.set_title('Factor Analysis Loadings (Varimax Rotation)\nEach row = one original feature, each column = one factor',
             color=NAVY, fontweight='bold', fontsize=12)
ax.set_xlabel('Factors')
ax.set_ylabel('Features')
plt.tight_layout()
plt.show()

# Factor naming
print("\nFACTOR NAMING (based on top-loading features):")
print("-" * 55)
factor_names = {
    'Factor_1': 'Financial Scale',
    'Factor_2': 'External Creditworthiness',
    'Factor_3': 'Employment & Age Stability',
    'Factor_4': 'Loan Leverage & Burden',
    'Factor_5': 'Credit History Depth',
    'Factor_6': 'Application Behaviour',
}
for i, col in enumerate(fa_loadings.columns[:6]):
    top3 = fa_loadings[col].abs().nlargest(3).index.tolist()
    name = factor_names.get(col, f'Factor {i+1}')
    print(f"  {col:12s} → '{name}'  |  Top features: {top3}")

# Factor 1: Financial Scale (AMT_CREDIT, AMT_GOODS_PRICE, AMT_ANNUITY)
# Factor 2: External Creditworthiness (EXT_SOURCE_1, 2, 3, EXT_SCORE_BLEND)
# Factor 3: Employment & Age Stability (AGE_YEARS, EMPLOYMENT_YEARS, DAYS_BIRTH)
# Factor 4: Loan Leverage & Burden (DEBT_TO_INCOME, ANNUITY_BURDEN)
# Factor 5: Credit History Depth (DAYS_CREDIT, bureau features)
# Factor 6: Application Behaviour (DAYS_REGISTRATION, DAYS_ID_PUBLISH)


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">For the heatmap, select features that load heavily on any factor: <code>for col in loadings.columns: tops.update(loadings[col].abs().nlargest(8).index)</code>. Sort features by their dominant factor: <code>loadings.abs().idxmax(axis=1)</code>.</span>
</div>

<div style="background: #F0E6F6; border-left: 5px solid #6C5B7B; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #6C5B7B;">👔 Manager’s Take</strong><br>
<span style="color: #333;">Factor names are the bridge between analytics and strategy. When you say “this applicant scores low on Financial Scale and External Credit Health,” a manager immediately understands the risk. When you say “PC2 is negative,” they don’t. Factor Analysis exists to make dimensionality reduction <em>communicable</em>.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 6: PCA vs FA: Head-to-Head</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">When to use which, and what each tells a manager</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Directly compare PCA and Factor Analysis on the same data. Understand their different goals, strengths, and when each is the right tool.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 10: Side-by-Side Loading Comparison</h3>
</div>

In [ ]:
# PCA vs FA — Side-by-Side Loading Comparison
# Build PCA loadings for first 3 PCs
pca_loadings = pd.DataFrame(
    pca_full.components_[:3].T,
    index=numeric_decorr_scaled.columns,
    columns=['PC1', 'PC2', 'PC3']
)

# Compare PC2 (usually financial scale) vs Factor_1
# Choose shared features with highest absolute loading in either
shared_feats = (pca_loadings['PC1'].abs()
                .add(fa_loadings['Factor_2'].abs(), fill_value=0)
                .nlargest(15).index.tolist())

compare_df = pd.DataFrame({
    'PCA PC1 Loading': pca_loadings.loc[shared_feats, 'PC1'],
    'FA Factor_2 Loading': fa_loadings.loc[shared_feats, 'Factor_2']
}).sort_values('PCA PC1 Loading', key=abs, ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, col, color, title in [
    (axes[0], 'PCA PC1 Loading', TEAL, 'PCA — PC1\n(Mixed loadings, hard to name)'),
    (axes[1], 'FA Factor_2 Loading', CORAL, 'Factor Analysis — Factor 2\n(Sparser, easier to interpret)')
]:
    vals = compare_df[col]
    bar_colors = [color if v > 0 else NAVY for v in vals]
    axes[0 if ax == axes[0] else 1].barh(compare_df.index, vals, color=bar_colors, edgecolor='white')
    axes[0 if ax == axes[0] else 1].axvline(0, color='black', linewidth=0.8)
    axes[0 if ax == axes[0] else 1].set_title(title, color=NAVY, fontweight='bold')
    axes[0 if ax == axes[0] else 1].set_xlabel('Loading Value')
    axes[0 if ax == axes[0] else 1].tick_params(axis='y', labelsize=9)

plt.suptitle('PCA vs Factor Analysis: Which is More Interpretable?',
             fontsize=13, color=NAVY, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey difference:")
print("  PCA PC1: all features contribute with mixed signs — hard to name")
print("  FA Factor 2: a few features dominate clearly — easier to label 'External Creditworthiness'")
print("  FA is preferred for executive communication; PCA is preferred as model input")


<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border: 2px solid #0B1F3F; padding: 20px 24px; border-radius: 10px; margin: 16px 0;">
<h3 style="color: #0B1F3F; margin-top: 0;">PCA vs Factor Analysis: The Decision Framework</h3>
<table style="width: 100%; border-collapse: collapse; margin-top: 12px;">
<tr style="background: #0B1F3F; color: white;">
<th style="padding: 10px; text-align: left; border: 1px solid #008C8C;">Criterion</th>
<th style="padding: 10px; text-align: center; border: 1px solid #008C8C;">PCA</th>
<th style="padding: 10px; text-align: center; border: 1px solid #008C8C;">Factor Analysis</th>
</tr>
<tr><td style="padding: 8px; border: 1px solid #ddd;"><strong>Goal</strong></td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">Compress data (retain variance)</td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">Discover latent constructs</td></tr>
<tr style="background: #E0F2F2;"><td style="padding: 8px; border: 1px solid #ddd;"><strong>Loadings</strong></td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">Mixed — all features contribute</td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">Sparse — each feature loads on ~1 factor</td></tr>
<tr><td style="padding: 8px; border: 1px solid #ddd;"><strong>Interpretability</strong></td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">Low (hard to name components)</td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">High (factors map to business concepts)</td></tr>
<tr style="background: #E0F2F2;"><td style="padding: 8px; border: 1px solid #ddd;"><strong>Best for</strong></td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">Pre-processing, input to models</td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">Strategy, segmentation, reporting</td></tr>
<tr><td style="padding: 8px; border: 1px solid #ddd;"><strong>Manager-friendly?</strong></td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">❌ Not easily</td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">✅ Yes, with good naming</td></tr>
<tr style="background: #E0F2F2;"><td style="padding: 8px; border: 1px solid #ddd;"><strong>Use in credit risk</strong></td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">Feed PCs into a model as features</td>
<td style="padding: 8px; border: 1px solid #ddd; text-align: center;">Segment borrowers, explain risk drivers</td></tr>
</table>
</div>

<div style="background: linear-gradient(135deg, #FFF8E1, #FFF3E0); border: 2px solid #B8953E; padding: 16px 20px; border-radius: 8px; margin: 16px 0;">
<strong style="color: #B8953E;">🤔 Think About It</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">You’ve now seen three ways to reduce dimensionality: variance filters, correlation screening, and PCA/FA. In a real project, you wouldn’t use all of them independently — you’d combine them. If you were building a credit-scoring model, what would your feature reduction pipeline look like? What order would you apply these steps in? Write your pipeline before reading Part 7.</p>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 7: Building the Reduced Feature Set</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Making final decisions and preparing for modelling</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Synthesise everything from this session into a final, reduced feature set. Document every decision and its rationale. This set will feed directly into Session 15’s models.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 11: Feature Reduction Summary</h3>
</div>

In [ ]:
# Feature Reduction Pipeline Summary
SEP = "─" * 68

pipeline_steps = [
    ("0. Original numeric columns",       numeric_imputed.shape[1] + len(high_miss_cols),  "Raw dataset"),
    ("1. After >40% missing dropped",     numeric_imputed.shape[1],                        "Housing/property features removed"),
    ("2. After NZV filter",               numeric_imputed_filtered.shape[1],               f"{len(nzv_cols)} FLAG_DOCUMENT_* removed"),
    ("3. After correlation screening",    numeric_decorr.shape[1],                         f"{len(to_drop)} redundant features dropped"),
    ("4a. PCA (90% variance)",            n_pca_90,                                        "Optimal for logistic regression"),
    ("4b. PCA (80% variance = FA input)", n_pca_80,                                        "Fewer components, more aggressive"),
    ("5. Factor Analysis",                n_factors,                                        "For segmentation & reporting"),
]

print("═" * 70)
print("  SESSION 14 — FEATURE REDUCTION PIPELINE")
print("═" * 70)
print(f"\n{'Step':<40s} {'Features':>10s}  {'Notes'}")
print(SEP)
for step, n, note in pipeline_steps:
    print(f"  {step:<38s} {n:>8d}  {note}")
print(SEP)

print("\nRECOMMENDATION FOR SESSION 15:")
print("  → Tree-based models (XGBoost, LightGBM): use decorrelated set (Step 3)")
print("    Trees handle many features natively and do NOT need PCA.")
print("  → Logistic Regression: use PCA set (Step 4a, 90% variance)")
print("    LR benefits from orthogonal, standardised inputs.")
print("  → Executive reporting: use FA factors (Step 5)")
print("    Named factors ('External Creditworthiness') communicate better than 'PC3'.")


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Simply track the column counts at each stage. The key insight for the recommendation: tree-based models (XGBoost) handle many features well; logistic regression benefits from PCA. Recommend carrying forward BOTH sets.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 12: Save the Reduced Feature Sets</h3>
</div>

In [ ]:
# Save Reduced Feature Sets for Session 15
import os

out_dir = r'C:\Users\DevA\OneDrive\Desktop\All in one - Desktop Files\MBA\Q8\AI ML\April 1 and 8'

# 1. Decorrelated feature set (for tree-based models)
decorr_out = numeric_decorr.copy()
decorr_out['TARGET'] = target.values
out_path_1 = os.path.join(out_dir, 's14_decorrelated_features.csv')
decorr_out.to_csv(out_path_1, index=False)
print(f"✅ Decorrelated features saved: {decorr_out.shape[1]-1} features + TARGET")
print(f"   Path: {out_path_1}")

# 2. PCA components (90% variance) for logistic regression
out_path_2 = os.path.join(out_dir, 's14_pca_features.csv')
pca_df.to_csv(out_path_2, index=False)
print(f"✅ PCA features saved: {n_pca_90} components + TARGET")
print(f"   Path: {out_path_2}")

print(f"\nSummary:")
print(f"  Decorrelated set : {decorr_out.shape[0]:,} rows × {decorr_out.shape[1]} columns")
print(f"  PCA set          : {pca_df.shape[0]:,} rows × {pca_df.shape[1]} columns")


In [ ]:
SEP68 = "─" * 68
print(f"\n┌{SEP68}┐")
for label, value in findings:
    print(f"│  {label:<30s} {value:>35s} │")
print(f"└{SEP68}┘")

print("\n✅ Session 14 Signal Extraction Complete!")
print("➡️  Next: Session 15 — Scalable Baselines (Logistic Regression & GBMs)")


<div style="background: linear-gradient(135deg, #0B1F3F 0%, #008C8C 100%); padding: 30px; border-radius: 12px; margin-top: 30px;">
<h2 style="color: white; font-family: Georgia, serif; margin: 0;">
✅ Session 14 Complete
</h2>
<p style="color: #B8953E; font-size: 1.15em; margin-top: 10px;">
You can now remove uninformative features, screen for redundancy, compress with PCA,
and discover latent constructs with Factor Analysis. Most importantly, you can explain
the difference between PCA and FA to a non-technical audience.
</p>
<p style="color: #B0D0D0; font-size: 1em; margin-top: 8px;">
<strong>Next Session Preview:</strong> In Session 15, we build our first models: regularised logistic
regression and gradient boosting machines. The two feature sets you saved today will go head-to-head.
Which representation produces better predictions? And are the predictions <em>calibrated</em> —
does a “20% default probability” actually mean 20%?
</p>
</div>